### Cell 1: ライブラリのインポート
このセルでは、データ分析、数値計算、ディープラーニングモデルの構築に必要なライブラリをまとめてインポートしています。

In [1]:
# === OSやファイル操作関連 ===
import os, json, joblib
from pathlib import Path

# === データ処理・数値計算関連 ===
import numpy as np  # 数値計算ライブラリ
import pandas as pd # データフレーム操作ライブラリ
import polars as pl # 高速なデータフレーム操作ライブラリ

# === 機械学習・ディープラーニング関連 ===
# scikit-learn: 機械学習の便利ツール群
from sklearn.model_selection import StratifiedGroupKFold     # 層化グループK-Fold交差検証
from sklearn.preprocessing import StandardScaler, LabelEncoder # 標準化とラベルエンコーディング
from sklearn.utils.class_weight import compute_class_weight  # クラス重みの計算

# TensorFlow/Keras: ディープラーニングフレームワーク
import tensorflow as tf
from tensorflow.keras.utils import Sequence, to_categorical, pad_sequences # データ供給、カテゴリ変換、パディング
from tensorflow.keras.models import Model, load_model                     # モデルの定義・読み込み
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, Activation, add, MaxPooling1D, Dropout,
    Bidirectional, LSTM, GlobalAveragePooling1D, Dense, Multiply, Reshape,
    Lambda, Concatenate, GRU, GaussianNoise
) # モデルを構成する様々なレイヤー
from tensorflow.keras.regularizers import l2             # L2正則化
from tensorflow.keras.optimizers import Adam             # 最適化アルゴリズム
from tensorflow.keras.callbacks import EarlyStopping     # 早期終了コールバック
from tensorflow.keras import backend as K                  # Kerasの低レベルAPI

# === その他 ===
import warnings
warnings.filterwarnings("ignore") # 不要な警告メッセージを非表示にする

from scipy.spatial.transform import Rotation as R # 回転を扱うためのSciPyの機能

import matplotlib.pyplot as plt # グラフ描画ライブラリ

2025-07-07 13:57:13.392046: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751896633.588391      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751896633.642722      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


### Cell 2: 乱数シードの固定
このセルでは、実験の再現性を確保するために、各種ライブラリの乱数シードを固定する関数を定義・実行しています。

In [2]:
import random

# 再現性のために乱数シードを固定する関数
def seed_everything(seed):
    # OSレベルのハッシュのランダム性を固定
    os.environ['PYTHONHASHSEED'] = str(seed)
    # Python標準の乱数を固定
    random.seed(seed)
    # NumPyの乱数を固定
    np.random.seed(seed)
    # TensorFlowのCPU/GPU乱数を固定
    tf.random.set_seed(seed)
    tf.experimental.numpy.random.seed(seed)
    # GPUの計算アルゴリズムを決定論的にする（再現性を高めるが、速度が低下する場合がある）
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

# シード値を42に設定して関数を実行
seed_everything(seed=42)

### Cell 3: グローバル変数とハイパーパラメータの設定
このセルでは、ノートブック全体で使用する定数やハイパーパラメータを設定しています。

In [3]:
# --- モード設定 ---
# Trueにすると学習モード、Falseにすると推論モードで実行される
TRAIN = False
# デバッグ用のゲート解析モード（今回は使われていない）
DEBUG_GATE = False

# --- パス設定 ---
RAW_DIR = Path("/kaggle/input/cmi-detect-behavior-with-sensor-data") # 入力データ（コンペ提供）のディレクトリ
PRETRAINED_DIR = Path("/kaggle/input/pretrained")      # 学習済みモデルや成果物を格納するディレクトリ
EXPORT_DIR = Path("./")                                             # 学習成果物の出力先ディレクトリ

# --- モデル・学習のハイパーパラメータ ---
BATCH_SIZE = 64          # バッチサイズ
PAD_PERCENTILE = 95      # パディングするシーケンス長のパーセンタイル点
LR_INIT = 5e-4           # 学習率の初期値
WD = 3e-3                # Weight Decay (L2正則化の強さ)
MIXUP_ALPHA = 0.4        # Mixupのパラメータα
EPOCHS = 160             # 最大エポック数
PATIENCE = 40            # EarlyStoppingのpatience（何エポック改善が見られなければ終了するか）
N_SPLITS = 5             # 交差検証の分割数

# --- 新機能に関するハイパーパラメータ ---
MASKING_PROB = 0.2      # 学習時にTOF/THMデータをランダムにマスク（ゼロ化）する確率
GATE_LOSS_WEIGHT = 0.15   # ゲートの補助損失に対する重み

print("▶ imports ready · tensorflow", tf.__version__)

▶ imports ready · tensorflow 2.18.0


### Cell 4: 特徴量エンジニアリング用関数
このセルでは、センサーデータから物理的な意味を持つ特徴量を計算するための関数を定義しています。

In [4]:
# 加速度データから重力成分を除去し、線形加速度を計算する関数
def remove_gravity_from_acc(acc_data, rot_data):
    acc_values = acc_data[['acc_x', 'acc_y', 'acc_z']].values
    quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values # クォータニオン(回転)データ
    linear_accel = np.zeros_like(acc_values)
    gravity_world = np.array([0, 0, 9.81]) # 世界座標系における重力ベクトル

    for i in range(len(acc_values)):
        # クォータニオンデータが無効な場合はスキップ
        if np.all(np.isnan(quat_values[i])) or np.all(np.isclose(quat_values[i], 0)):
            linear_accel[i, :] = acc_values[i, :]
            continue
        try:
            # クォータニオンから回転オブジェクトを生成
            rotation = R.from_quat(quat_values[i])
            # 世界座標系の重力ベクトルを、センサーの座標系に変換
            gravity_sensor_frame = rotation.apply(gravity_world, inverse=True)
            # 観測された加速度から重力成分を引く
            linear_accel[i, :] = acc_values[i, :] - gravity_sensor_frame
        except ValueError:
             linear_accel[i, :] = acc_values[i, :]
    return linear_accel

# クォータニオンデータから角速度を計算する関数
def calculate_angular_velocity_from_quat(rot_data, time_delta=1/200): # サンプリングレートは200Hz
    quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
    angular_vel = np.zeros((len(quat_values), 3))

    for i in range(len(quat_values) - 1):
        q_t, q_t_plus_dt = quat_values[i], quat_values[i+1] # 連続する2つのクォータニオン
        if np.all(np.isnan(q_t)) or np.all(np.isnan(q_t_plus_dt)): continue
        try:
            rot_t = R.from_quat(q_t)
            rot_t_plus_dt = R.from_quat(q_t_plus_dt)
            # 2つの回転の差分を計算
            delta_rot = rot_t.inv() * rot_t_plus_dt
            # 差分回転を回転ベクトルに変換し、時間で割って角速度を求める
            angular_vel[i, :] = delta_rot.as_rotvec() / time_delta
        except ValueError: pass
    return angular_vel

# 連続するタイムステップ間の回転角度（角距離）を計算する関数
def calculate_angular_distance(rot_data):
    quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
    angular_dist = np.zeros(len(quat_values))

    for i in range(len(quat_values) - 1):
        q1, q2 = quat_values[i], quat_values[i+1]
        if np.all(np.isnan(q1)) or np.all(np.isnan(q2)): continue
        try:
            r1, r2 = R.from_quat(q1), R.from_quat(q2)
            # 2つの回転の差分を計算
            relative_rotation = r1.inv() * r2
            # 差分回転を回転ベクトルに変換し、その大きさ（ノルム）を角距離とする
            angular_dist[i] = np.linalg.norm(relative_rotation.as_rotvec())
        except ValueError: pass
    return angular_dist

### Cell 5: モデル構築用のカスタム関数/レイヤー
このセルでは、ディープラーニングモデルを構築するために使われる、再利用可能なカスタムのブロックや関数を定義しています。

In [5]:
# --- TensorFlow/Kerasのテンソル操作用ヘルパー関数 ---
def time_sum(x): return K.sum(x, axis=1) # 時間軸（第1軸）で合計する
def squeeze_last_axis(x): return tf.squeeze(x, axis=-1) # 最後の軸を削除する
def expand_last_axis(x): return tf.expand_dims(x, axis=-1) # 最後の軸を追加する

# Squeeze-and-Excitation (SE) ブロック
# チャネルごとの特徴量の重要度を学習し、適応的に特徴量を強調する
def se_block(x, reduction=8):
    ch = x.shape[-1] # チャネル数を取得
    # Global Average Poolingで各チャネルの情報を集約
    se = GlobalAveragePooling1D()(x)
    # 全結合層で重要度を計算
    se = Dense(ch // reduction, activation='relu')(se)
    se = Dense(ch, activation='sigmoid')(se) # sigmoidで0〜1の重みに変換
    se = Reshape((1, ch))(se) # 元のテンソルの形に合わせる
    # 元の特徴量に計算した重要度を掛け合わせる
    return Multiply()([x, se])

# Residual SE-CNN ブロック
# CNNにSEブロックと残差接続(Shortcut)を組み合わせた、高性能な基本ブロック
def residual_se_cnn_block(x, filters, kernel_size, pool_size=2, drop=0.3, wd=1e-4):
    shortcut = x # 入力を保存（残差接続用）

    # 2層のCNN
    for _ in range(2):
        x = Conv1D(filters, kernel_size, padding='same', use_bias=False,
                   kernel_regularizer=l2(wd))(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)

    x = se_block(x) # SEブロックを適用

    # 入力と出力のチャネル数が異なる場合、ショートカットも変換する
    if shortcut.shape[-1] != filters:
        shortcut = Conv1D(filters, 1, padding='same', use_bias=False,
                          kernel_regularizer=l2(wd))(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # 出力とショートカットを加算（残差接続）
    x = add([x, shortcut])
    x = Activation('relu')(x)
    x = MaxPooling1D(pool_size)(x) # プーリングで次元削減
    x = Dropout(drop)(x) # ドロップアウトで過学習抑制
    return x

# アテンション機構レイヤー
# 時系列データの中で、予測に重要なタイムステップに注目（重み付け）する
def attention_layer(inputs):
    # 全結合層で各タイムステップの重要度スコアを計算
    score = Dense(1, activation='tanh')(inputs)
    score = Lambda(squeeze_last_axis)(score)
    # softmaxでスコアを正規化し、重み（合計1）に変換
    weights = Activation('softmax')(score)
    weights = Lambda(expand_last_axis)(weights)
    # 元の入力に重みを掛け合わせる
    context = Multiply()([inputs, weights])
    # 重み付けされた特徴量を時間軸で合計し、文脈ベクトルを生成
    context = Lambda(time_sum)(context)
    return context

### Cell 6: カスタムデータジェネレータ 
このセルでは、学習中にMixup、マスキング、クラス重み付けなどを行う高機能なデータジェネレータを定義しています。

In [6]:
# KerasのSequenceを継承したカスタムデータジェネレータ
class GatedMixupGenerator(Sequence):
    # 初期化
    def __init__(self, X, y, batch_size, imu_dim, class_weight=None, alpha=0.2, masking_prob=0.0):
        self.X, self.y = X, y                 # データとラベル
        self.batch = batch_size               # バッチサイズ
        self.imu_dim = imu_dim                # IMU特徴量の次元数
        self.class_weight = class_weight      # クラスごとの重み
        self.alpha = alpha                    # Mixupのパラメータ
        self.masking_prob = masking_prob      # ToF/THMデータをマスクする確率
        self.indices = np.arange(len(X))      # データのインデックス配列

    # 1エポックあたりのバッチ数を返す
    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch))

    # 1バッチ分のデータを生成するメソッド
    def __getitem__(self, i):
        # 現在のバッチに対応するインデックスを取得
        idx = self.indices[i*self.batch:(i+1)*self.batch]
        Xb, yb = self.X[idx].copy(), self.y[idx].copy()

        # --- クラスの不均衡を考慮するためのサンプルごとの重みを計算 ---
        sample_weights = np.ones(len(Xb), dtype='float32')
        if self.class_weight:
            y_integers = yb.argmax(axis=1) # one-hotから整数ラベルに変換
            sample_weights = np.array([self.class_weight[i] for i in y_integers])

        # --- ゲーティング機構のための処理 ---
        # ゲートの目標値(1: ToF/THMがON, 0: OFF)
        gate_target = np.ones(len(Xb), dtype='float32')
        # 指定した確率でToF/THMデータをマスク(0で埋める)し、ゲートの目標値を0にする
        if self.masking_prob > 0:
            for i in range(len(Xb)):
                if np.random.rand() < self.masking_prob:
                    Xb[i, :, self.imu_dim:] = 0 # IMU以降の次元(ToF/THM)を0にする
                    gate_target[i] = 0.0

        # --- Mixup処理 ---
        if self.alpha > 0:
            # ベータ分布からMixupの混合比率(lam)をサンプリング
            lam = np.random.beta(self.alpha, self.alpha)
            # 混ぜ合わせるためにインデックスをシャッフル
            perm = np.random.permutation(len(Xb))
            # データ、ラベル、ゲート目標値、サンプル重みをそれぞれ混ぜ合わせる
            X_mix = lam * Xb + (1 - lam) * Xb[perm]
            y_mix = lam * yb + (1 - lam) * yb[perm]
            gate_target_mix = lam * gate_target + (1 - lam) * gate_target[perm]
            sample_weights_mix = lam * sample_weights + (1 - lam) * sample_weights[perm]
            # モデルの出力名に合わせて辞書形式で返す
            return X_mix, {'main_output': y_mix, 'tof_gate': gate_target_mix}, sample_weights_mix

        # Mixupしない場合は、通常のデータを返す
        return Xb, {'main_output': yb, 'tof_gate': gate_target}, sample_weights

    # 1エポック終了時に呼ばれるメソッド
    def on_epoch_end(self):
        # 次のエポックのためにデータのインデックスをシャッフルする
        np.random.shuffle(self.indices)

### Cell 7: 2ブランチ・ゲート付きモデル構築関数
このセルでは、このノートブックの心臓部であるディープラーニングモデル全体の構造を定義しています。

In [7]:
# ゲーティング機構を持つ2ブランチモデルを構築する関数
def build_gated_two_branch_model(pad_len, imu_dim, tof_dim, n_classes, wd=1e-4):
    # --- 入力レイヤー ---
    # 全ての特徴量(IMU + ToF/THM)を受け取る
    inp = Input(shape=(pad_len, imu_dim+tof_dim))

    # --- 2ブランチへの分割 ---
    # Lambdaレイヤーを使って、入力をIMU部分とToF/THM部分に分ける
    imu = Lambda(lambda t: t[:, :, :imu_dim])(inp)
    tof = Lambda(lambda t: t[:, :, imu_dim:])(inp)

    # --- ブランチ1: IMU ---
    # Residual SE-CNNブロックを2つ重ねて特徴抽出
    x1 = residual_se_cnn_block(imu, 64, 3, drop=0.1, wd=wd)
    x1 = residual_se_cnn_block(x1, 128, 5, drop=0.1, wd=wd)

    # --- ブランチ2: ToF/THM ---
    # こちらはシンプルなCNNブロックで特徴抽出
    x2_base = Conv1D(64, 3, padding='same', use_bias=False, kernel_regularizer=l2(wd))(tof)
    x2_base = BatchNormalization()(x2_base); x2_base = Activation('relu')(x2_base)
    x2_base = MaxPooling1D(2)(x2_base); x2_base = Dropout(0.2)(x2_base)
    x2_base = Conv1D(128, 3, padding='same', use_bias=False, kernel_regularizer=l2(wd))(x2_base)
    x2_base = BatchNormalization()(x2_base); x2_base = Activation('relu')(x2_base)
    x2_base = MaxPooling1D(2)(x2_base); x2_base = Dropout(0.2)(x2_base)

    # --- ゲーティング機構 ---
    # ToFデータ全体からゲート値を計算
    gate_input = GlobalAveragePooling1D()(tof)
    gate_input = Dense(16, activation='relu')(gate_input)
    # ゲート値(0〜1)を出力。補助損失の計算対象とするため、レイヤーに 'tof_gate' と命名
    gate = Dense(1, activation='sigmoid', name='tof_gate')(gate_input)

    # ToF/THMブランチの出力にゲート値を乗算し、情報の流量を制御
    x2 = Multiply()([x2_base, gate])

    # --- ブランチの結合と後段処理 ---
    # 2つのブランチの特徴量を結合
    merged = Concatenate()([x1, x2])
    # Bi-LSTMとBi-GRUで時間的特徴を抽出
    xa = Bidirectional(LSTM(128, return_sequences=True, kernel_regularizer=l2(wd)))(merged)
    xb = Bidirectional(GRU(128, return_sequences=True, kernel_regularizer=l2(wd)))(merged)
    # ノイズを加えてDenseレイヤーを通したパスも追加（多様性のため）
    xc = GaussianNoise(0.09)(merged)
    xc = Dense(16, activation='elu')(xc)
    # 3つのパスを結合
    x = Concatenate()([xa, xb, xc])
    x = Dropout(0.4)(x)
    # アテンション機構で重要なタイムステップに注目
    x = attention_layer(x)
    # 最終的な分類を行う全結合層
    for units, drop in [(256, 0.5), (128, 0.3)]:
        x = Dense(units, use_bias=False, kernel_regularizer=l2(wd))(x)
        x = BatchNormalization()(x); x = Activation('relu')(x)
        x = Dropout(drop)(x)
    # 主出力(ジェスチャーの分類確率)。損失計算のために 'main_output' と命名
    out = Dense(n_classes, activation='softmax', name='main_output', kernel_regularizer=l2(wd))(x)

    # 入力と2つの出力（主出力、ゲート出力）を持つモデルを定義して返す
    return Model(inputs=inp, outputs=[out, gate])

In [8]:
# if DEBUG_GATE and not TRAIN:
#     print("▶ GATE DEBUG MODE – preparing data to analyze trained models...")
    
#     # --- 1. 学習時と全く同じデータ準備プロセスを実行 ---
#     df = pd.read_csv(RAW_DIR / "train.csv")
#     train_dem_df = pd.read_csv(RAW_DIR / "train_demographics.csv")
#     df = pd.merge(df, train_dem_df, on='subject', how='left')
#     le = LabelEncoder()
#     df['gesture_int'] = le.fit_transform(df['gesture'])
#     print("  Calculating engineered features...")
#     df['acc_mag'] = np.sqrt(df['acc_x']**2 + df['acc_y']**2 + df['acc_z']**2)
#     df['rot_angle'] = 2 * np.arccos(df['rot_w'].clip(-1, 1))
#     df['acc_mag_jerk'] = df.groupby('sequence_id')['acc_mag'].diff().fillna(0)
#     df['rot_angle_vel'] = df.groupby('sequence_id')['rot_angle'].diff().fillna(0)
#     cols_for_stats = ['acc_mag', 'rot_angle', 'acc_mag_jerk', 'rot_angle_vel']
#     for col in cols_for_stats:
#         df[f'{col}_skew'] = df.groupby('sequence_id')[col].transform('skew')
#         df[f'{col}_kurt'] = df.groupby('sequence_id')[col].transform(pd.Series.kurtosis)
    
#     # --- [修正点] 成果物の読み込み元から特徴量リストを取得 ---
#     final_feature_cols = np.load(PRETRAINED_DIR / "feature_cols.npy", allow_pickle=True).tolist()
    
#     print("  Building sequences...")
#     seq_gp = df.groupby('sequence_id')
#     X_list_unscaled, y_list_int, groups_list, lens = [], [], [], []
#     for seq_id, seq_df in seq_gp:
#         seq_df_copy = seq_df.copy()
#         for i in range(1, 6):
#             pixel_cols_tof = [f"tof_{i}_v{p}" for p in range(64)]
#             tof_sensor_data = seq_df_copy[pixel_cols_tof].replace(-1, np.nan)
#             seq_df_copy[f'tof_{i}_mean'] = tof_sensor_data.mean(axis=1)
#             seq_df_copy[f'tof_{i}_std']  = tof_sensor_data.std(axis=1)
#             seq_df_copy[f'tof_{i}_min']  = tof_sensor_data.min(axis=1)
#             seq_df_copy[f'tof_{i}_max']  = tof_sensor_data.max(axis=1)
#         mat_unscaled = seq_df_copy[final_feature_cols].ffill().bfill().fillna(0).values.astype('float32')
#         X_list_unscaled.append(mat_unscaled)
#         y_list_int.append(seq_df_copy['gesture_int'].iloc[0])
#         groups_list.append(seq_df_copy['subject'].iloc[0])
#         lens.append(len(mat_unscaled))

#     print("  Loading scaler and padding sequences...")
#     # --- [修正点] 読み込み元を PRETRAINED_DIR に変更 ---
#     scaler = joblib.load(PRETRAINED_DIR / "scaler.pkl") 
#     pad_len = int(np.load(PRETRAINED_DIR / "sequence_maxlen.npy"))
    
#     X_scaled_list = [scaler.transform(x_seq) for x_seq in X_list_unscaled]
#     del X_list_unscaled
#     X = pad_sequences(X_scaled_list, maxlen=pad_len, padding='post', truncating='post', dtype='float32')
#     del X_scaled_list
#     y_stratify = np.array(y_list_int)
#     groups = np.array(groups_list)

#     # --- 2. CV分割を再現し、各フォールドのモデルを分析 ---
#     sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
#     custom_objs = {
#         'time_sum': time_sum, 'squeeze_last_axis': squeeze_last_axis, 'expand_last_axis': expand_last_axis,
#         'se_block': se_block, 'residual_se_cnn_block': residual_se_cnn_block, 'attention_layer': attention_layer,
#     }

#     for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y_stratify, groups)):
#         print(f"\n===== Analyzing FOLD {fold+1}/{N_SPLITS} =====")
#         X_val = X[val_idx]
        
#         # --- [修正点] 読み込み元を PRETRAINED_DIR に変更 ---
#         model_path = PRETRAINED_DIR / f"gesture_model_fold_{fold}.h5"
#         if not model_path.exists():
#             print(f"Model file not found: {model_path}. Skipping this fold.")
#             continue
#         model = load_model(model_path, compile=False, custom_objects=custom_objs)

#         if 'tof_gate' not in [layer.name for layer in model.layers]:
#             print("Error: 'tof_gate' layer not found in the model.")
#             break
#         gate_output = model.get_layer('tof_gate').output
#         debug_model = Model(inputs=model.input, outputs=[model.output, gate_output])

#         print("  Predicting on validation data to get gate values...")
#         _, gate_values = debug_model.predict(X_val, batch_size=64) # BATCH_SIZE
#         gate_values = gate_values.flatten()

#         print(f"  Gate Value Stats: Mean={np.mean(gate_values):.4f}, Std={np.std(gate_values):.4f}, Min={np.min(gate_values):.4f}, Max={np.max(gate_values):.4f}")

#         plt.figure(figsize=(10, 5))
#         plt.hist(gate_values, bins=50, range=(0, 1), color='skyblue', edgecolor='black')
#         plt.title(f"Fold {fold+1} Gate Value Distribution", fontsize=16)
#         plt.xlabel("Gate Value (0 = TOF/THM Off, 1 = TOF/THM On)", fontsize=12)
#         plt.ylabel("Frequency", fontsize=12)
#         plt.grid(axis='y', alpha=0.75)
#         plt.show()
    
#     print("\n✔ Gate analysis finished.")

######################### TRAINフラグがTrueの場合、学習プロセスを実行 #########################
if TRAIN:
    print("▶ 学習モードを開始します – データセットを読み込んでいます…")
    # --- 1. データの読み込みと初期前処理 ---
    # 学習用データと被験者のデモグラフィック（属性）データを読み込む
    df = pd.read_csv(RAW_DIR / "train.csv")
    train_dem_df = pd.read_csv(RAW_DIR / "train_demographics.csv")

    # 'subject'列をキーとして、2つのデータフレームを結合
    df = pd.merge(df, train_dem_df, on='subject', how='left')

    # ジェスチャーの文字列ラベル（例: "Wave"）を整数（例: 0, 1, 2...）に変換する
    le = LabelEncoder()
    df['gesture_int'] = le.fit_transform(df['gesture'])

    # 後で推論時に使うために、ラベルエンコーダーが覚えたクラス名（ジェスチャー名の一覧）を保存
    np.save(EXPORT_DIR / "gesture_classes.npy", le.classes_)

    # --- 2. 物理ベースの特徴量エンジニアリング ---
    print("  物理特徴量を計算中: 重力加速度の除去と線形加速度の算出...")
    # 'sequence_id' ごとにグループ化し、各シーケンスで定義済みの関数を使って物理特徴量を計算
    # 加速度データから重力成分を除去
    linear_accel_list = [pd.DataFrame(remove_gravity_from_acc(group[['acc_x', 'acc_y', 'acc_z']], group[['rot_x', 'rot_y', 'rot_z', 'rot_w']]), columns=['linear_acc_x', 'linear_acc_y', 'linear_acc_z'], index=group.index) for _, group in df.groupby('sequence_id')]
    # 計算結果を元のデータフレームに結合
    df = pd.concat([df, pd.concat(linear_accel_list)], axis=1)
    # 線形加速度の大きさとその変化量（Jerk）を計算
    df['linear_acc_mag'] = np.sqrt(df['linear_acc_x']**2 + df['linear_acc_y']**2 + df['linear_acc_z']**2)
    df['linear_acc_mag_jerk'] = df.groupby('sequence_id')['linear_acc_mag'].diff().fillna(0)

    print("  物理特徴量を計算中: クォータニオンから角速度と角距離を算出...")
    # クォータニオンデータから角速度を計算
    angular_vel_list = [pd.DataFrame(calculate_angular_velocity_from_quat(group[['rot_x', 'rot_y', 'rot_z', 'rot_w']]), columns=['angular_vel_x', 'angular_vel_y', 'angular_vel_z'], index=group.index) for _, group in df.groupby('sequence_id')]
    df = pd.concat([df, pd.concat(angular_vel_list)], axis=1)
    # クォータニオンデータから角距離を計算
    angular_dist_list = [pd.DataFrame(calculate_angular_distance(group[['rot_x', 'rot_y', 'rot_z', 'rot_w']]), columns=['angular_distance'], index=group.index) for _, group in df.groupby('sequence_id')]
    df = pd.concat([df, pd.concat(angular_dist_list)], axis=1)

    # --- 3. 最終的な特徴量の定義 ---
    # モデルに入力する特徴量をリストとして定義する
    # IMU関連: 物理ベースの特徴量と回転(rot)に関する特徴量
    imu_cols_base = ['linear_acc_x', 'linear_acc_y', 'linear_acc_z'] + [c for c in df.columns if c.startswith('rot_')]
    imu_engineered = ['linear_acc_mag', 'linear_acc_mag_jerk', 'angular_vel_x', 'angular_vel_y', 'angular_vel_z', 'angular_distance']
    imu_cols = list(dict.fromkeys(imu_cols_base + imu_engineered))
    # 温度(thm)関連
    thm_cols_original = [c for c in df.columns if c.startswith('thm_')]
    # ToF関連: 生の64ピクセルデータではなく、統計量（平均、標準偏差など）を使う
    tof_aggregated_cols_template = []
    for i in range(1, 6): tof_aggregated_cols_template.extend([f'tof_{i}_mean', f'tof_{i}_std', f'tof_{i}_min', f'tof_{i}_max'])

    # 上記の全ての特徴量を結合して最終的な特徴量リストとする
    final_feature_cols = imu_cols + thm_cols_original + tof_aggregated_cols_template
    imu_dim_final = len(imu_cols)
    tof_thm_aggregated_dim_final = len(thm_cols_original) + len(tof_aggregated_cols_template)
    # 後で使うために特徴量リストを保存
    np.save(EXPORT_DIR / "feature_cols.npy", np.array(final_feature_cols))

    # --- 4. シーケンスデータの構築とスケーリング ---
    print("  シーケンスデータを構築しています...")
    # タイムスタンプごとのフラットなDataFrameを、シーケンスごとのリスト形式に変換する
    seq_gp = df.groupby('sequence_id')
    X_list_unscaled, y_list_int, groups_list, lens = [], [], [], []
    for seq_id, seq_df in seq_gp:
        seq_df_copy = seq_df.copy()
        # 各シーケンス内でToFの統計量を計算
        for i in range(1, 6):
            pixel_cols = [f"tof_{i}_v{p}" for p in range(64)]
            tof_data = seq_df_copy[pixel_cols].replace(-1, np.nan)
            seq_df_copy[f'tof_{i}_mean'] = tof_data.mean(axis=1)
            seq_df_copy[f'tof_{i}_std'] = tof_data.std(axis=1)
            seq_df_copy[f'tof_{i}_min'] = tof_data.min(axis=1)
            seq_df_copy[f'tof_{i}_max'] = tof_data.max(axis=1)
        # 欠損値を前方・後方から補完し、それでも残る欠損は0で埋める
        X_list_unscaled.append(seq_df_copy[final_feature_cols].ffill().bfill().fillna(0).values.astype('float32'))
        y_list_int.append(seq_df_copy['gesture_int'].iloc[0])      # 各シーケンスのラベルを追加
        groups_list.append(seq_df_copy['subject'].iloc[0])       # 各シーケンスの被験者IDを追加
        lens.append(len(seq_df_copy))                              # 各シーケンスの長さを記録

    print("  StandardScaler（標準化）を学習させています...")
    # 全てのシーケンスの全タイムステップを結合し、それらを使ってスケーラーを学習させる
    all_steps_concatenated = np.concatenate(X_list_unscaled, axis=0)
    scaler = StandardScaler().fit(all_steps_concatenated)
    # 学習済みのスケーラーを保存
    joblib.dump(scaler, EXPORT_DIR / "scaler.pkl")

    print("  各シーケンスをスケーリングし、長さを揃えるパディング処理を行っています...")
    # 学習済みスケーラーを各シーケンスに適用
    X_scaled_list = [scaler.transform(x_seq) for x_seq in X_list_unscaled]
    # 全シーケンス長の95パーセンタイル点をパディング長とする
    pad_len = int(np.percentile(lens, PAD_PERCENTILE))
    np.save(EXPORT_DIR / "sequence_maxlen.npy", pad_len) # パディング長を保存
    # 全てのシーケンスが同じ長さ(pad_len)になるように、後ろを0で埋める（または切り捨てる）
    X = pad_sequences(X_scaled_list, maxlen=pad_len, padding='post', truncating='post', dtype='float32')
    # ラベルとグループ情報をNumPy配列に変換し、ラベルはワンホットエンコーディングする
    y_stratify, groups, y = np.array(y_list_int), np.array(groups_list), to_categorical(y_list_int, num_classes=len(le.classes_))

    # --- 5. Stratified Group K-Foldによる交差検証と学習 ---
    print("  層化グループK-Fold交差検証による学習を開始します...")
    # Stratified: 各分割でラベルの比率を維持 / Group: 同じ被験者が学習用と検証用に分かれないようにする
    sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    # Out-of-Fold (OOF)予測を格納するための配列を初期化
    oof_preds = np.zeros_like(y, dtype='float32')

    # K-Foldのループを開始
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y_stratify, groups)):
        print(f"\n===== フォールド {fold+1}/{N_SPLITS} の学習 =====")
        # 学習用データと検証用データに分割
        X_tr, X_val, y_tr, y_val = X[train_idx], X[val_idx], y[train_idx], y[val_idx]
        
        # モデルの構築
        model = build_gated_two_branch_model(pad_len, imu_dim_final, tof_thm_aggregated_dim_final, len(le.classes_), wd=WD)
        
        # モデルのコンパイル設定
        model.compile(optimizer=Adam(LR_INIT),
                      # 損失関数: 主出力とゲート出力のそれぞれに異なる損失関数を指定
                      loss={'main_output': tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), 'tof_gate': tf.keras.losses.BinaryCrossentropy()},
                      # 損失の重み: 主損失を1.0、補助損失であるゲート損失をGATE_LOSS_WEIGHTに設定
                      loss_weights={'main_output': 1.0, 'tof_gate': GATE_LOSS_WEIGHT},
                      # 評価指標: 主出力の正解率を監視
                      metrics={'main_output': 'accuracy'})
        
        # クラスの不均衡を是正するため、各クラスの重みを計算
        class_weight_dict = dict(enumerate(compute_class_weight('balanced', classes=np.arange(len(le.classes_)), y=y_tr.argmax(1))))
        
        # データジェネレータの準備
        # 学習用: MixUpやマスキング、クラス重みを適用
        train_gen = GatedMixupGenerator(X_tr, y_tr, batch_size=BATCH_SIZE, imu_dim=imu_dim_final, class_weight=class_weight_dict, alpha=MIXUP_ALPHA, masking_prob=MASKING_PROB)
        # 検証用: データ拡張は行わない
        val_gen = GatedMixupGenerator(X_val, y_val, batch_size=BATCH_SIZE, imu_dim=imu_dim_final)
        
        # 早期終了(EarlyStopping)コールバック: 検証データの精度が一定期間改善しなければ学習を打ち切る
        cb = EarlyStopping(patience=PATIENCE, restore_best_weights=True, verbose=0, monitor='val_main_output_accuracy', mode='max')
        
        # モデルの学習を実行
        model.fit(train_gen, epochs=EPOCHS, validation_data=val_gen, callbacks=[cb], verbose=0)
        
        # 最良の重みが復元されたモデルを保存
        model.save(EXPORT_DIR / f"gesture_model_fold_{fold}.h5")
        
        # 検証データで予測を行い、OOF予測を保存
        preds_val, _ = model.predict(X_val)
        oof_preds[val_idx] = preds_val

    # --- 6. OOFスコアの計算 ---
    # OOF予測は、各データが「自身を含まないデータで学習されたモデル」によって予測された結果の集まり。
    # これにより、モデルの汎化性能をより正確に評価できる。
    print("\n✔ 学習が完了しました。")
    print("  Out-of-Fold予測を使って評価スコアを計算します...")
    
    # コンペティションの公式評価指標を計算するためのクラスをインポート
    from cmi_2025_metric_copy_for_import import CompetitionMetric
    
    # one-hot形式から整数ラベルに戻す
    true_oof_int = y.argmax(1)
    pred_oof_int = oof_preds.argmax(1)
    
    # 評価指標を計算
    h_f1_oof = CompetitionMetric().calculate_hierarchical_f1(
        pd.DataFrame({'gesture': le.classes_[true_oof_int]}),
        pd.DataFrame({'gesture': le.classes_[pred_oof_int]}))
    print(f"Overall OOF H‑F1 Score = {h_f1_oof:.4f}")

########################## TRAINフラグがFalseの場合、推論準備の処理を実行 #########################
else:
    # --- 1. 学習済み成果物（Artifacts）の読み込み ---
    print(f"▶ 推論モードを開始します – 学習済み成果物を'{PRETRAINED_DIR}'から読み込んでいます…")
    
    # 学習時に使用した特徴量のリストを読み込む
    # => これにより、推論時にも学習時と全く同じ特徴量を同じ順序で使うことができる
    final_feature_cols = np.load(PRETRAINED_DIR / "feature_cols.npy", allow_pickle=True).tolist()
    
    # 学習時に決定した、シーケンスの長さを揃えるためのパディング長を読み込む
    pad_len = int(np.load(PRETRAINED_DIR / "sequence_maxlen.npy"))
    
    # 学習済みStandardScaler（標準化器）を読み込む
    # => 学習データと同じ平均・標準偏差でテストデータを変換するために必須
    scaler = joblib.load(PRETRAINED_DIR / "scaler.pkl")
    
    # ジェスチャーのクラス名（文字列のリスト）を読み込む
    # => モデルの出力（例: 0, 1, 2...）を実際のジェスチャー名（例: "Wave"）に戻すために使用
    gesture_classes = np.load(PRETRAINED_DIR / "gesture_classes.npy", allow_pickle=True)

    # --- 2. カスタムオブジェクトの定義 ---
    # モデルの独自コンポーネント（自作レイヤーや関数）を定義する
    # Kerasはモデルを保存する際、これらの自作関数の定義までは保存しないため、
    # モデルを正しく読み込むには、名前と関数をマッピングした辞書を渡す必要がある
    custom_objs = {
        'time_sum': time_sum, 
        'squeeze_last_axis': squeeze_last_axis, 
        'expand_last_axis': expand_last_axis,
        'se_block': se_block, 
        'residual_se_cnn_block': residual_se_cnn_block, 
        'attention_layer': attention_layer,
    }

    # --- 3. 学習済みモデルの読み込み ---
    # アンサンブル推論のために、交差検証で作成した全てのモデルをリストに格納する
    models = []
    print(f"  アンサンブル推論のために{N_SPLITS}個のモデルを読み込んでいます...")
    
    # forループで5つのフォールドモデルを順番に読み込む
    for fold in range(N_SPLITS):
        model_path = PRETRAINED_DIR / f"gesture_model_fold_{fold}.h5"
        
        # モデルを読み込む
        # compile=False: 推論時には再学習しないため、コンパイルは不要で高速化できる
        # custom_objects: 上で定義した独自コンポーネントの辞書を渡す
        model = load_model(model_path, compile=False, custom_objects=custom_objs)
        
        # 読み込んだモデルをリストに追加
        models.append(model)
        
    print("  モデル、スケーラー、特徴量リスト、パディング長の読み込みが完了し、評価準備が整いました。")

▶ 推論モードを開始します – 学習済み成果物を'/kaggle/input/pretrained'から読み込んでいます…
  アンサンブル推論のために5個のモデルを読み込んでいます...


I0000 00:00:1751896646.006087      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


  モデル、スケーラー、特徴量リスト、パディング長の読み込みが完了し、評価準備が整いました。


In [9]:
# --- TTA用のハイパーパラメータ ---
TTA_STEPS = 1           # TTAの実行回数　デフォ＝10
TTA_NOISE_STDDEV = 0.005  # 入力データに加えるノイズの標準偏差 デフォ＝0.01

# Kaggleの評価サーバーから呼び出される推論関数
def predict(sequence: pl.DataFrame, demographics: pl.DataFrame) -> str:
    df_seq = sequence.to_pandas()

    # --- 1. 特徴量エンジニアリング（学習時と全く同じ処理） ---
    # (重力除去、角速度計算など、学習時と同じ特徴量を追加) 
    linear_accel = remove_gravity_from_acc(df_seq, df_seq)
    df_seq['linear_acc_x'], df_seq['linear_acc_y'], df_seq['linear_acc_z'] = linear_accel[:, 0], linear_accel[:, 1], linear_accel[:, 2]
    df_seq['linear_acc_mag'] = np.sqrt(df_seq['linear_acc_x']**2 + df_seq['linear_acc_y']**2 + df_seq['linear_acc_z']**2)
    df_seq['linear_acc_mag_jerk'] = df_seq['linear_acc_mag'].diff().fillna(0)
    angular_vel = calculate_angular_velocity_from_quat(df_seq)
    df_seq['angular_vel_x'], df_seq['angular_vel_y'], df_seq['angular_vel_z'] = angular_vel[:, 0], angular_vel[:, 1], angular_vel[:, 2]
    df_seq['angular_distance'] = calculate_angular_distance(df_seq)
    for i in range(1, 6):
        pixel_cols = [f"tof_{i}_v{p}" for p in range(64)]; tof_data = df_seq[pixel_cols].replace(-1, np.nan)
        df_seq[f'tof_{i}_mean'], df_seq[f'tof_{i}_std'], df_seq[f'tof_{i}_min'], df_seq[f'tof_{i}_max'] = tof_data.mean(axis=1), tof_data.std(axis=1), tof_data.min(axis=1), tof_data.max(axis=1)

    # --- 2. 前処理（スケーリングとパディング） ---
    mat_unscaled = df_seq[final_feature_cols].ffill().bfill().fillna(0).values.astype('float32')
    mat_scaled = scaler.transform(mat_unscaled)
    pad_input = pad_sequences([mat_scaled], maxlen=pad_len, padding='post', truncating='post', dtype='float32')

    # --- 3. Test-Time Augmentation (TTA) とアンサンブル予測 ---
    all_tta_predictions = []
    for _ in range(TTA_STEPS):
        # 1回目はノイズなし、2回目以降は入力にガウスノイズを加える
        if TTA_STEPS > 1 and _ > 0:
             noisy_input = pad_input + tf.random.normal(shape=tf.shape(pad_input), mean=0.0, stddev=TTA_NOISE_STDDEV)
        else:
             noisy_input = pad_input

        # 5つのフォールドモデルでそれぞれ予測
        all_fold_predictions = []
        for model in models:
            # 主出力(ジェスチャー確率)のみを取得
            main_preds, _ = model.predict(noisy_input, verbose=0)
            all_fold_predictions.append(main_preds)

        # フォールド間の予測を平均（アンサンブル）
        avg_fold_prediction = np.mean(all_fold_predictions, axis=0)
        all_tta_predictions.append(avg_fold_prediction)

    # --- 4. 最終予測 ---
    # TTAの結果をさらに平均
    final_avg_prediction = np.mean(all_tta_predictions, axis=0)

    # 最も確率の高いクラスのインデックスを取得
    idx = int(final_avg_prediction.argmax())

    # 対応するジェスチャークラス名を返す
    return str(gesture_classes[idx])

In [10]:
# TRAIN=False の場合のみ実行
if not TRAIN:
    # Kaggle提供の推論サーバーライブラリをインポート
    import kaggle_evaluation.cmi_inference_server
    # 上で定義したpredict関数を渡して、サーバーを初期化
    inference_server = kaggle_evaluation.cmi_inference_server.CMIInferenceServer(predict)

    # Kaggleのコンペ提出環境（本番）であるかを環境変数でチェック
    if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
        # 本番環境なら、サーバーを起動して提出を待つ
        inference_server.serve()
    else:
        # 本番環境でない場合（手元でのテストなど）、ローカルでテスト用のゲートウェイを起動
        inference_server.run_local_gateway(
            data_paths=(
                '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv',
                '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv',
            )
        )

2025-07-07 13:57:31.451716: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
I0000 00:00:1751896652.360289      58 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-07-07 13:57:37.342266: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:var